In [0]:
%pip install torch>=2.0.0 torchvision>=0.15.0 lightgbm fastf1>=3.6.0

In [0]:
%run /Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/model-training-and-testing/models

In [0]:
%run /Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/model-training-and-testing/dataloader

In [0]:
%run /Users/s3yuen@uwaterloo.ca/F1-Race-Engineer/model-training-and-testing/testing

In [0]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pandas as pd
from pyspark.sql import SparkSession
from sklearn.metrics import mean_absolute_error
import joblib
import os
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# 1. Load and preprocess data from Unity Catalog
df = spark.table('workspace.f1_racing_laptime_pred.silver_training').toPandas()
df = assign_stints(df)

# 2. Continuous features
cont_features = ['LapNumber', 'TyreLife', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall',
                 'TrackTemp', 'WindDirection', 'WindSpeed', 'Circuit_CircuitLength',
                 'Circuit_Number_of_Laps', 'Circuit_NumberOfTurns', 'Circuit_AverageAngleAbs',
                 'Circuit_AverageAngle', 'GapToLeader', 'GapToAhead',
                 'GapToBehind', 'status_1', 'status_12', 'status_124',
                 'status_21', 'status_24', 'status_4',
                 'status_41',  # Add this line
                 "TimeSinceLastWeatherMeasurement"]

# 3. Create dataset and dataloader
dataset = StintDataset(df, cont_features)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

# 4. Model
n_drivers = df['Driver_idx'].nunique()
n_teams = df['Team_idx'].nunique()
n_tyres = int((df['Compound'] - 1).max() + 1)  # compute after converting to 0-based indices
n_modes = df['mode'].nunique()
n_cont_features = len(cont_features)

In [0]:
model = StintTransformer(n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres,
                         n_modes=n_modes, n_cont_features=n_cont_features)


In [0]:
model = StintLSTM(n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres,
                  n_modes=n_modes, n_cont_features=n_cont_features)

In [0]:
model = StintGRU(n_drivers=n_drivers, n_teams=n_teams, n_tyres=n_tyres,
                  n_modes=n_modes, n_cont_features=n_cont_features)

In [0]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [0]:
def train_and_save(model, dataloader, optimizer, criterion, device, num_epochs=19, save_path='stint_transformer_model.pth', use_plateau: bool = True, plateau_kwargs: dict = None):
    """Train provided model and save weights to `save_path`.

    Returns the trained model.
    """
    model.to(device)
    print(f"Starting training for {num_epochs} epochs")
    scheduler = None
    if use_plateau:
        plateau_kwargs = plateau_kwargs or {}
        # sensible defaults: halve LR on plateau, wait 3 epochs
        defaults = dict(mode='min', factor=0.5, patience=3)
        merged = {**defaults, **plateau_kwargs}
        try:
            scheduler = ReduceLROnPlateau(optimizer, **merged)
            print(f"ReduceLROnPlateau scheduler enabled with params: {merged}")
        except TypeError as e:
            # Some torch versions may not support newer kwargs; fall back to a minimal scheduler
            print(f"Warning: ReduceLROnPlateau init failed ({e}); scheduler disabled.")
            scheduler = None
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0

        for cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, lap_time, mask in dataloader:
            # Move to device
            cont_feats = cont_feats.to(device)
            driver_idx = driver_idx.to(device)
            team_idx = team_idx.to(device)
            tyre_idx = tyre_idx.to(device)
            mode_idx = mode_idx.to(device)
            lap_time = lap_time.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()

            # Forward pass
            outputs = model(cont_feats, driver_idx, team_idx, tyre_idx, mode_idx, mask)

            # Mask out padded values
            loss = criterion(outputs[~mask], lap_time[~mask])

            # Backward pass
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        epoch_metric = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_metric:.4f}")
        # Step scheduler on the monitored metric (training loss here)
        if scheduler is not None:
            scheduler.step(epoch_metric)
            # print current LR(s)
            lrs = {i: g['lr'] for i, g in enumerate(optimizer.param_groups)}
            print(f"Learning rates: {lrs}")

    torch.save(model.state_dict(), save_path)
    model.eval()
    print(f"Model saved to {save_path}")
    return model


In [0]:
trained = train_and_save(model, dataloader, optimizer, criterion, device, num_epochs=25, save_path='stint_model.pth')

In [0]:
# Run predictions on silver_validating and write to gold_predicted_validation
mae, out_df = predict_and_evaluate(
    'stint_model.pth', 
    device, 
    'workspace.f1_racing_laptime_pred.silver_validating'
)

# Print error report from the gold table
errors, distribution = print_error_report()